# Data Analysis

## Part 1: SA4 spatial join exploration

Initial exploration of the SA4 boundaries and a first spatial join of the raw charger coordinates.

In [1]:
import geopandas as gpd

In [2]:
gdf = gpd.read_file(
    "../data/raw/SA4_2026_AUST_SHP_GDA2020/SA4_2026_AUST_GDA2020.shp"
)

print(gdf.head())
print(gdf.columns)

  SA4_CODE26               SA4_NAME26 CHG_FLAG26  CHG_LBL26 GCC_CODE26  \
0        101           Capital Region          0  No change      1RNSW   
1        102            Central Coast          0  No change      1GSYD   
2        103             Central West          0  No change      1RNSW   
3        104  Coffs Harbour - Grafton          0  No change      1RNSW   
4        105       Far West and Orana          0  No change      1RNSW   

       GCC_NAME26 STE_CODE26       STE_NAME26 AUS_CODE26 AUS_NAME26  \
0     Rest of NSW          1  New South Wales        AUS  Australia   
1  Greater Sydney          1  New South Wales        AUS  Australia   
2     Rest of NSW          1  New South Wales        AUS  Australia   
3     Rest of NSW          1  New South Wales        AUS  Australia   
4     Rest of NSW          1  New South Wales        AUS  Australia   

    AREASQKM26                                           geometry  
0   51896.2445  MULTIPOLYGON (((150.08176 -36.377, 150.08159

In [3]:
print(gdf.crs)

EPSG:7844


In [4]:
import pandas as pd

ev = pd.read_csv("../data/raw/ev_20251216.csv")

print(ev.head())
print(ev.columns.tolist())
print(ev.shape)

   OBJECTID Station_name                               Station_address  \
0       NaN          NaN                          , Muswellbrook, 2333   
1       NaN          NaN               01 Wallgrove Road, Sydney, 2766   
2       NaN          NaN  1 - 7 Ross St, Wilcannia NSW 2836, Australia   
3       NaN          NaN                    1 Balfour St, Sydney, 2070   
4       NaN          NaN                     1 Bay Ln, Byron Bay, 2481   

    Operator  Number_of_plugs Charger_Type Charger_rating   Latitude  \
0       EVUp                2           AC          22 kW -32.262242   
1         BP                4           DC         150 kW -33.811004   
2       NRMA                4           DC          50 kW -30.511874   
3  Chargefox                7           AC          22 kW -33.774101   
4      Tesla                2           AC          19 kW -28.641819   

    Longitude                        LGANAME PCODE  \
0  150.890139     Muswellbrook Shire Council  2333   
1  150.849597 

In [5]:
ev_gdf = gpd.GeoDataFrame(
    ev,
    geometry=gpd.points_from_xy(ev["Longitude"], ev["Latitude"]),
    crs="EPSG:4326"
)

print(ev_gdf.head())
print(ev_gdf.crs)

   OBJECTID Station_name                               Station_address  \
0       NaN          NaN                          , Muswellbrook, 2333   
1       NaN          NaN               01 Wallgrove Road, Sydney, 2766   
2       NaN          NaN  1 - 7 Ross St, Wilcannia NSW 2836, Australia   
3       NaN          NaN                    1 Balfour St, Sydney, 2070   
4       NaN          NaN                     1 Bay Ln, Byron Bay, 2481   

    Operator  Number_of_plugs Charger_Type Charger_rating   Latitude  \
0       EVUp                2           AC          22 kW -32.262242   
1         BP                4           DC         150 kW -33.811004   
2       NRMA                4           DC          50 kW -30.511874   
3  Chargefox                7           AC          22 kW -33.774101   
4      Tesla                2           AC          19 kW -28.641819   

    Longitude                        LGANAME PCODE  \
0  150.890139     Muswellbrook Shire Council  2333   
1  150.849597 

In [6]:
ev_gdf = ev_gdf.to_crs(gdf.crs)

print(ev_gdf.crs)
print(gdf.crs)

EPSG:7844
EPSG:7844


In [7]:
joined = gpd.sjoin(
    ev_gdf,
    gdf[["SA4_CODE26", "SA4_NAME26", "STE_NAME26", "geometry"]],
    how="left",
    predicate="within"
)

print(joined[
    ["Station_name", "Latitude", "Longitude",
     "SA4_CODE26", "SA4_NAME26", "STE_NAME26"]
].head(10))

print("Original EV rows:", len(ev_gdf))
print("Joined rows:", len(joined))
print("No SA4 match:", joined["SA4_CODE26"].isna().sum())

  Station_name   Latitude   Longitude SA4_CODE26  \
0          NaN -32.262242  150.890139        106   
1          NaN -33.811004  150.849597        116   
2          NaN -30.511874  151.669395        110   
3          NaN -33.774101  151.167035        121   
4          NaN -28.641819  153.613633        112   
5          NaN -33.883504  151.194433        117   
6          NaN -28.276903  153.577078        112   
7          NaN -33.841037  151.242245        121   
8          NaN -33.841418  151.242631        121   
9          NaN -33.871368  151.213741        117   

                          SA4_NAME26       STE_NAME26  
0        Hunter Valley exc Newcastle  New South Wales  
1                 Sydney - Blacktown  New South Wales  
2         New England and North West  New South Wales  
3  Sydney - North Sydney and Hornsby  New South Wales  
4                   Richmond - Tweed  New South Wales  
5      Sydney - City and Inner South  New South Wales  
6                   Richmond - Twee

In [8]:
unmatched = joined[joined["SA4_CODE26"].isna()]

print(unmatched[
    ["Station_name", "Station_address",
     "Latitude", "Longitude", "LGANAME", "PCODE"]
])

     Station_name                               Station_address  Latitude  \
1833          NaN  1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia -33.80467   

      Longitude                   LGANAME PCODE  
1833  151.25284  Northern Beaches Council  2093  


In [9]:
# Get the unmatched EV charger
point = unmatched.geometry.iloc[0]

# Convert both datasets to a projected CRS for distance calculation
point_proj = gpd.GeoSeries([point], crs="EPSG:7844").to_crs("EPSG:7856")
sa4_proj = gdf.to_crs("EPSG:7856")

# Calculate distance to every SA4 polygon
distances = sa4_proj.geometry.distance(point_proj.iloc[0])

# Find the nearest SA4
nearest_idx = distances.idxmin()

print("Nearest SA4:", sa4_proj.loc[nearest_idx, "SA4_NAME26"])
print("State:", sa4_proj.loc[nearest_idx, "STE_NAME26"])
print("Distance:", round(distances.loc[nearest_idx], 2), "metres")

Nearest SA4: Sydney - Northern Beaches
State: New South Wales
Distance: 1.77 metres


In [10]:
# Fill the unmatched charger with its nearest SA4
joined.loc[unmatched.index, "SA4_CODE26"] = gdf.loc[nearest_idx, "SA4_CODE26"]
joined.loc[unmatched.index, "SA4_NAME26"] = gdf.loc[nearest_idx, "SA4_NAME26"]
joined.loc[unmatched.index, "STE_NAME26"] = gdf.loc[nearest_idx, "STE_NAME26"]

# Check the result
print(joined.loc[unmatched.index, [
    "Station_address",
    "SA4_CODE26",
    "SA4_NAME26",
    "STE_NAME26"
]])

print("No SA4 match:", joined["SA4_CODE26"].isna().sum())

                                   Station_address SA4_CODE26  \
1833  1 Sandy Bay Rd\nClontarf NSW 2093\nAustralia        122   

                     SA4_NAME26       STE_NAME26  
1833  Sydney - Northern Beaches  New South Wales  
No SA4 match: 0


## Part 2: Raw EV charger data profile

This part profiles `data/raw/ev_20251216.csv` as downloaded, before any cleaning. Every column is read as text with `keep_default_na=False`, so empty strings, whitespace and formatting are shown exactly as they appear in the source file.

In [11]:
import pandas as pd
import geopandas as gpd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

raw = pd.read_csv("../data/raw/ev_20251216.csv", dtype=str, keep_default_na=False)
print("Shape:", raw.shape)
raw.head()

Shape: (1958, 12)


,OBJECTID,Station_name,Station_address,Operator,Number_of_plugs,Charger_Type,Charger_rating,Latitude,Longitude,LGANAME,PCODE,Source
0,,,", Muswellbrook, 2333",EVUp,2,AC,22 kW,-32.26224229,150.8901391,Muswellbrook Shire Council,2333,Existing Destination Chargers
1,,,"01 Wallgrove Road, Sydney, 2766",BP,4,DC,150 kW,-33.81100405,150.8495966,Blacktown City Council,2766,Existing Fast Chargers
2,,,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,4,DC,50 kW,-30.5118739,151.669395,Central Darling Shire Council,2350,TfNSW Regional
3,,,"1 Balfour St, Sydney, 2070",Chargefox,7,AC,22 kW,-33.77410115,151.167035,Ku-ring-gai Council,2070,Existing Destination Chargers
4,,,"1 Bay Ln, Byron Bay, 2481",Tesla,2,AC,19 kW,-28.64181886,153.6136326,Byron Shire Council,2481,Existing Destination Chargers


### 2.1 Column completeness and text formatting

In [12]:
def text_profile(df):
    """Count empty values, distinct values, surrounding whitespace and line breaks per column."""
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "column": col,
            "empty": (s.str.strip() == "").sum(),
            "empty_pct": round((s.str.strip() == "").mean() * 100, 1),
            "distinct_non_empty": s[s.str.strip() != ""].nunique(),
            "leading_trailing_space": (s != s.str.strip()).sum(),
            "contains_newline": s.str.contains("\n").sum(),
        })
    return pd.DataFrame(rows).set_index("column")

text_profile(raw)

,empty,empty_pct,distinct_non_empty,leading_trailing_space,contains_newline
column,,,,,
OBJECTID,1837,93.8,121,0,0
Station_name,1438,73.4,477,20,0
Station_address,0,0.0,1924,3,733
Operator,0,0.0,50,55,0
Number_of_plugs,0,0.0,18,0,0
Charger_Type,0,0.0,3,0,0
Charger_rating,0,0.0,46,0,0
Latitude,0,0.0,1937,0,0
Longitude,0,0.0,1935,0,0


**Findings**

- `OBJECTID` is empty in 1837 of 1958 rows (93.8%).
- `Station_name` is empty in 1438 rows (73.4%).
- `LGANAME`, `PCODE` and `Source` are each empty in 121 rows (6.2%).
- Surrounding whitespace: `Operator` 55 rows, `Station_name` 20, `Station_address` 3.
- `Station_address` contains line breaks in 733 rows.
- `Number_of_plugs`, `Charger_Type`, `Charger_rating`, `Latitude` and `Longitude` have no empty values.

### 2.2 Missing-value pattern

Check whether the empty `LGANAME`, `PCODE` and `Source` values are related to the rows that have an `OBJECTID`.

In [13]:
has_objectid = raw["OBJECTID"] != ""

print("Rows with OBJECTID:", has_objectid.sum())
print("Row index range of rows with OBJECTID:",
      raw.index[has_objectid].min(), "-", raw.index[has_objectid].max())

missing_pattern = pd.DataFrame({
    col: raw[col].groupby(has_objectid).apply(lambda s: (s == "").sum())
    for col in ["LGANAME", "PCODE", "Source", "Station_name"]
})
missing_pattern.index = missing_pattern.index.map({True: "has OBJECTID", False: "no OBJECTID"})
missing_pattern["rows"] = has_objectid.value_counts().rename(
    {True: "has OBJECTID", False: "no OBJECTID"})
missing_pattern

Rows with OBJECTID: 121
Row index range of rows with OBJECTID: 983 - 1103


,LGANAME,PCODE,Source,Station_name,rows
OBJECTID,,,,,
no OBJECTID,0,0,0,1339,1837
has OBJECTID,121,121,121,99,121


In [14]:
pd.crosstab(raw["Charger_Type"],
            has_objectid.map({True: "has OBJECTID", False: "no OBJECTID"}),
            margins=True)

OBJECTID,has OBJECTID,no OBJECTID,All
Charger_Type,,,
AC,17,1410,1427
DC,6,427,433
Upcoming,98,0,98
All,121,1837,1958


In [15]:
# Share of rows with a Station_name, by charger type
(raw["Station_name"].str.strip() != "").groupby(raw["Charger_Type"]).agg(["sum", "count", "mean"])

,sum,count,mean
Charger_Type,,,
AC,515,1427,0.360897
DC,1,433,0.002309
Upcoming,4,98,0.040816


**Findings**

- The 121 rows with an `OBJECTID` are contiguous in the file (row index 983-1103).
- All 121 of these rows have empty `LGANAME`, `PCODE` and `Source`. None of the 1837 rows without an `OBJECTID` is empty in these three columns.
- All 98 `Upcoming` rows have an `OBJECTID`. The remaining 23 rows with an `OBJECTID` are 17 AC and 6 DC.
- `Station_name` is filled for 36.1% of AC rows, 0.2% of DC rows (1 of 433) and 4.1% of `Upcoming` rows.

### 2.3 Charger_Type

In [16]:
raw["Charger_Type"].value_counts()

Charger_Type
AC          1427
DC           433
Upcoming      98
Name: count, dtype: int64

**Findings**

- Three values: `AC` 1427, `DC` 433, `Upcoming` 98. No spelling variants.
- `Upcoming` is stored in the same column as `AC`/`DC`, so `Upcoming` rows have no AC/DC value.

### 2.4 Charger_rating formats

Classify each rating string by its written format.

In [17]:
def rating_format(value):
    if pd.Series([value]).str.fullmatch(r"\d+(\.\d+)? kW").iloc[0]:
        return "number + ' kW'"
    if value.isdigit():
        return "number only"
    if "&" in value:
        return "combined (e.g. 2x350kW & 2x175kW)"
    return value

rating_fmt = raw["Charger_rating"].map(rating_format)
pd.crosstab(rating_fmt, raw["Charger_Type"], margins=True)

Charger_Type,AC,DC,Upcoming,All
Charger_rating,,,,
AC,522,0,0,522
combined (e.g. 2x350kW & 2x175kW),0,5,94,99
number + ' kW',888,427,0,1315
number only,17,1,4,22
All,1427,433,98,1958


In [18]:
# All non-standard rating values
raw.loc[rating_fmt != "number + ' kW'", "Charger_rating"].value_counts()

Charger_rating
AC                   522
2x350kW & 2x175kW     85
22                    16
2x350kW & 6x175kW     14
7                      5
50                     1
Name: count, dtype: int64

In [19]:
raw["Charger_rating"].value_counts()

Charger_rating
22 kW                634
AC                   522
6 kW                 112
50 kW                 86
2x350kW & 2x175kW     85
75 kW                 80
7 kW                  66
25 kW                 52
175 kW                46
150 kW                44
11 kW                 31
125 kW                19
130 kW                16
22                    16
60 kW                 15
19 kW                 14
2x350kW & 6x175kW     14
120 kW                11
250 kW                10
47 kW                  8
350 kW                 7
40 kW                  7
180 kW                 6
3 kW                   6
8 kW                   5
17 kW                  5
7                      5
14 kW                  4
23 kW                  4
160 kW                 3
20 kW                  3
42 kW                  3
24 kW                  2
80 kW                  2
100 kW                 2
200 kW                 2
13 kW                  2
30 kW                  1
185 kW                 1
5 kW      

**Findings**

- 1315 rows use the format `N kW`.
- 522 rows (all AC) contain the text `AC` instead of a power value.
- 22 rows give a number without a unit: `22` (16), `7` (5), `50` (1).
- 99 rows contain combined values: `2x350kW & 2x175kW` (85) and `2x350kW & 6x175kW` (14). 94 of them are `Upcoming` and 5 are DC.
- All 121 rows with an `OBJECTID` use either the number-only or the combined format. All 1837 rows without an `OBJECTID` use `N kW` or `AC`.
- The column is text and cannot be used as a numeric power value without parsing.

### 2.5 Operator

All operator values, sorted alphabetically so that similar spellings sit next to each other. `repr` makes trailing spaces visible.

In [20]:
operator_counts = raw["Operator"].value_counts().rename_axis("Operator").reset_index(name="rows")
operator_counts["repr"] = operator_counts["Operator"].map(repr)
operator_counts["rows_with_OBJECTID"] = operator_counts["Operator"].map(
    raw[has_objectid]["Operator"].value_counts()).fillna(0).astype(int)
operator_counts.sort_values("Operator", key=lambda s: s.str.lower()).reset_index(drop=True)

,Operator,rows,repr,rows_with_OBJECTID
0,360 EV Charge,5,'360 EV Charge',0
1,Alchemy Charge,1,'Alchemy Charge',0
2,Ampol,32,'Ampol',0
3,AXCharge,1,'AXCharge',0
4,BMW,1,'BMW',0
5,BP,32,'BP',0
6,BP Australia,28,'BP Australia ',28
7,CasaCharge,5,'CasaCharge',0
8,Charge Hub,7,'Charge Hub',0
9,Charge OS,2,'Charge OS',0


In [21]:
# Values that only differ by case, spaces or punctuation
key = raw["Operator"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
variants = raw.groupby(key)["Operator"].unique()
variants[variants.map(len) > 1]

Operator
chargehub              [ChargeHub, Charge Hub]
nonnetworked    [Non-networked, Non-Networked]
Name: Operator, dtype: object

In [22]:
# Length of operator values, split by whether the row has an OBJECTID
op_len = raw["Operator"].str.len()
op_len.groupby(has_objectid.map({True: "has OBJECTID", False: "no OBJECTID"})).describe()

,count,mean,std,min,25%,50%,75%,max
OBJECTID,,,,,,,,
has OBJECTID,121.0,12.297521,2.242189,3.0,13.0,13.0,13.0,13.0
no OBJECTID,1837.0,7.501361,3.544554,2.0,5.0,7.0,9.0,28.0


**Findings**

- 50 distinct values.
- Trailing space: `'BP Australia '` (28 rows) and `'Tesla Motors '` (27 rows).
- Values that differ only in case or spacing: `Non-networked` / `Non-Networked`, `ChargeHub` / `Charge Hub`.
- Values where one is the other followed by further words: `Tesla` / `Tesla Motors`, `BP` / `BP Australia`, `Evie` / `Evie Networks`, `NRMA` / `NRMA Electric`, `PLUS ES` / `PLUS ES Manag`, `Viva Energy A` / `Viva Energy Australia`.
- In the 121 rows with an `OBJECTID`, no operator value is longer than 13 characters, and 109 are exactly 13 characters long (including trailing spaces), e.g. `Fast Cities A`, `Energy Austra`, `University of`, `PLUS ES Manag`, `Viva Energy A`. In rows without an `OBJECTID`, values are up to 28 characters long.

### 2.6 Station_name

In [23]:
name = raw["Station_name"]
print("Empty:", (name.str.strip() == "").sum())
print("Leading/trailing spaces:", (name != name.str.strip()).sum())
print(name[name != name.str.strip()].map(repr).head(10).tolist())
print()
same_as_operator = raw[name == raw["Operator"]]
print("Station_name identical to Operator:", len(same_as_operator))
same_as_operator[["OBJECTID", "Station_name", "Operator", "Charger_Type", "Station_address"]]

Empty: 1438
Leading/trailing spaces: 20
["'Angullong Wines '", "'Landcare & Sustainable Living Centre '", "'St Kilda Street '", "'Darling Square Car Park '", "'Lake Conjola Bowling and Recreation Club '", "'Twin Towns Services Club '", "'Twin Towns Services Club '", "'Moruya Golf Club '", "'Corowa RSL Club '", "'Antikvorem '"]

Station_name identical to Operator: 8


,OBJECTID,Station_name,Operator,Charger_Type,Station_address
999,136,EVE Australia,EVE Australia,AC,"52 Pembroke St, Ashfield NSW 2131, Australia"
1035,411,Evie Networks,Evie Networks,DC,"92 Houston Rd, Kingsford NSW 2032, Australia"
1050,493,EVX,EVX,AC,"60 Frenchs Rd, Willoughby NSW 2068, Australia"
1055,521,EVE Australia,EVE Australia,AC,1A Keith St Dulwich Hill NSW
1059,542,EVX,EVX,AC,"3 Hilltop Cres, Fairlight NSW 2094, Australia"
1074,605,EVX,EVX,AC,"251 Edgecliff Rd, Woollahra NSW 2025, Australia"
1094,747,PLUS ES,PLUS ES,AC,"180 Darling St, Balmain NSW 2041, Australia"
1099,813,Chargefox,Chargefox,AC,"44 Station St, Wickham NSW 2293, Australia"


**Findings**

- Empty in 1438 rows; 432 of the 433 DC rows have no name.
- 20 values have surrounding whitespace.
- 8 rows, all with an `OBJECTID`, have a `Station_name` identical to their `Operator` (e.g. `EVX` / `EVX`, `Chargefox` / `Chargefox`).

### 2.7 Station_address

In [24]:
addr = raw["Station_address"]
print("Contains newline:", addr.str.contains("\n").sum())
print("Leading/trailing spaces:", (addr != addr.str.strip()).sum())
print("Contains a 4-digit number starting with 2:", addr.str.contains(r"\b2\d{3}\b").sum(),
      "of", len(addr))
print("Distinct addresses:", addr.nunique())
print("Starting with a comma (empty street part):", addr.str.strip().str.startswith(",").sum(),
      addr[addr.str.strip().str.startswith(",")].tolist())
print("Without a 4-digit number starting with 2:", addr[~addr.str.contains(r"\b2\d{3}\b")].map(repr).tolist())
print()
print("Examples:")
for a in addr.sample(8, random_state=1):
    print(repr(a))

Contains newline: 733
Leading/trailing spaces: 3
Contains a 4-digit number starting with 2: 1954 of 1958
Distinct addresses: 1924
Starting with a comma (empty street part): 1 [', Muswellbrook, 2333']
Without a 4-digit number starting with 2: ["'10 Burkinshaw St, Barooga, 3644'", "'1A Keith St Dulwich Hill NSW'", "'65A Barwan Street Narrabri NSW'", "'Burkinshaw St\\nBarooga NSW 3644\\nAustralia'"]

Examples:
'McFarlane St, Sydney, 2160'
'697 Wollombi Rd\nBroke NSW 2330\nAustralia'
'154 Beach Rd, Batemans Bay NSW 2536'
'53 Macquarie Rd\nCardiff NSW 2285\nAustralia'
'76 Wingewarra St, Dubbo , 2830'
'Lauder St, Tumbarumba, 2653'
'13 Southey St, Jerilderie NSW 2716, Australia'
'293 Belmore Rd, Sydney, 2210'


**Findings**

- 733 rows separate address parts with line breaks (e.g. `697 Wollombi Rd\nBroke NSW 2330\nAustralia`); the others use commas in different patterns (e.g. `2 Stewart St, Lithgow, 2790` and `13 Southey St, Jerilderie NSW 2716, Australia`).
- 1924 distinct values for 1958 rows. The same site can appear with different formatting (see 2.11).
- 3 values have surrounding whitespace.
- 1 value has an empty street part: `, Muswellbrook, 2333`.
- 2 values have no postcode (`1A Keith St Dulwich Hill NSW`, `65A Barwan Street Narrabri NSW`); 2 have postcode `3644` (Barooga).

### 2.8 PCODE

In [25]:
pcode = raw.loc[raw["PCODE"] != "", "PCODE"]
print("Non-empty:", len(pcode))
print("Exactly 4 digits:", pcode.str.fullmatch(r"\d{4}").sum())
print("Other formats:", pcode[~pcode.str.fullmatch(r"\d{4}")].tolist())

Non-empty: 1837
Exactly 4 digits: 1827
Other formats: ['NSW 2500', 'NSW 2500', 'NSW 2291', 'NSW 2481', 'NSW 2515', 'NSW 2500', 'NSW 2324', 'NSW 2502', 'NSW 2303', 'NSW 2481']


**Findings**

- 1837 non-empty values; 1827 are exactly 4 digits and 10 are prefixed with `NSW ` (e.g. `NSW 2500`).
- 121 empty values (the rows with an `OBJECTID`, see 2.2).

### 2.9 Number_of_plugs

In [26]:
plugs = raw["Number_of_plugs"].astype(int)
print(plugs.describe())
plugs.value_counts().sort_index()

count    1958.000000
mean        2.873340
std         2.423335
min         1.000000
25%         2.000000
50%         2.000000
75%         4.000000
max        35.000000
Name: Number_of_plugs, dtype: float64


Number_of_plugs
1     416
2     855
3      64
4     451
5       9
6      63
7       1
8      46
9       3
10     12
11      1
12     23
15      3
16      3
18      1
20      5
26      1
35      1
Name: count, dtype: int64

In [27]:
plugs.groupby(raw["Charger_Type"]).describe()

,count,mean,std,min,25%,50%,75%,max
Charger_Type,,,,,,,,
AC,1427.0,2.400140,2.036229,1.0,1.0,2.0,3.0,35.0
DC,433.0,3.829099,2.916013,1.0,2.0,4.0,4.0,20.0
Upcoming,98.0,5.540816,2.257489,2.0,4.0,4.0,6.0,15.0


**Findings**

- Complete; all values are integers from 1 to 35 (median 2). No zero or negative values.

### 2.10 Coordinates

In [28]:
lat = raw["Latitude"].astype(float)
lon = raw["Longitude"].astype(float)
print("Latitude range:", lat.min(), "to", lat.max())
print("Longitude range:", lon.min(), "to", lon.max())
print("Non-numeric coordinates:",
      pd.to_numeric(raw["Latitude"], errors="coerce").isna().sum(),
      pd.to_numeric(raw["Longitude"], errors="coerce").isna().sum())

Latitude range: -37.11146251 to -28.1685927
Longitude range: 141.4601038 to 153.6158749
Non-numeric coordinates: 0 0


**Findings**

- All coordinates are numeric and fall within latitude -37.11 to -28.17 and longitude 141.46 to 153.62.
- No missing or out-of-range values.

### 2.11 Duplicates

Exact duplicates across all columns, and rows that share identical coordinates.

In [29]:
print("Exact duplicate rows (all 12 columns):", raw.duplicated().sum())

coord_cols = ["Latitude", "Longitude"]
shared = raw[raw.duplicated(coord_cols, keep=False)].sort_values(coord_cols)
print("Rows sharing identical coordinates:", len(shared),
      "in", shared.groupby(coord_cols).ngroups, "groups")
shared[["OBJECTID", "Operator", "Charger_Type", "Charger_rating",
        "Number_of_plugs", "Latitude", "Longitude", "Station_address"]]

Exact duplicate rows (all 12 columns): 0
Rows sharing identical coordinates: 38 in 18 groups


,OBJECTID,Operator,Charger_Type,Charger_rating,Number_of_plugs,Latitude,Longitude,Station_address
700,,Exploren,AC,AC,2,-29.869128,150.5714344,74 Maitland St\nBingara NSW 2404\nAustralia
1069,576,Non-Networked,AC,22,2,-29.869128,150.5714344,"74 Maitland St, Bingara NSW 2404, Australia"
815,,Exploren,AC,AC,2,-30.88653,153.0379572,Buchanan Dr\nSouth West Rocks NSW 2431\nAustralia
1010,200,Exploren,AC,22,2,-30.88653,153.0379572,"Buchanan Dr, South West Rocks NSW 2431, Australia"
980,,Non-networked,AC,AC,1,-32.7165581,151.2572876,"771 Hermitage Rd, Pokolbin NSW 2320"
1688,,Non-networked,AC,AC,2,-32.7165581,151.2572876,771 Hermitage Rd\nPokolbin NSW 2320\nAustralia
555,,Chargefox,AC,22 kW,2,-32.9230228,151.7572335,44 Station St\nWickham NSW 2293\nAustralia
1099,813,Chargefox,AC,22,2,-32.9230228,151.7572335,"44 Station St, Wickham NSW 2293, Australia"
981,,Non-networked,AC,AC,1,-33.3931163,151.3286306,"33 Gugandi Rd, Narara NSW 2250"
1493,,EVSE,AC,AC,3,-33.3931163,151.3286306,33 Gugandi Rd\nNarara NSW 2250\nAustralia


In [30]:
# For each shared-coordinate group: how many rows have an OBJECTID
group_summary = shared.assign(has_objectid=shared["OBJECTID"] != "").groupby(coord_cols).agg(
    rows=("Operator", "size"),
    rows_with_objectid=("has_objectid", "sum"),
    operators=("Operator", lambda s: " | ".join(s)),
    types=("Charger_Type", lambda s: " | ".join(s)),
    ratings=("Charger_rating", lambda s: " | ".join(s)),
    plugs=("Number_of_plugs", lambda s: " | ".join(s)),
)
print(group_summary["rows_with_objectid"].value_counts().rename("groups"))
group_summary

rows_with_objectid
1    14
0     4
Name: groups, dtype: int64


,,rows,rows_with_objectid,operators,types,ratings,plugs
Latitude,Longitude,,,,,,
-29.869128,150.5714344,2,1,Exploren | Non-Networked,AC | AC,AC | 22,2 | 2
-30.88653,153.0379572,2,1,Exploren | Exploren,AC | AC,AC | 22,2 | 2
-32.7165581,151.2572876,2,0,Non-networked | Non-networked,AC | AC,AC | AC,1 | 2
-32.9230228,151.7572335,2,1,Chargefox | Chargefox,AC | AC,22 kW | 22,2 | 2
-33.3931163,151.3286306,2,0,Non-networked | EVSE,AC | AC,AC | AC,1 | 3
-33.7949453,151.188291,2,0,Non-networked | Tesla,AC | DC,AC | 175 kW,2 | 8
-33.8073934,151.200067,2,1,EVX | EVX,AC | AC,22 kW | 22,2 | 2
-33.8556219,151.0767955,2,1,Engie | Tesla Motors,DC | Upcoming,150 kW | 2x350kW & 6x175kW,2 | 8
-33.8584276,151.186886,2,1,PLUS ES | PLUS ES,AC | AC,7 | 7 kW,1 | 1


**Findings**

- 0 exact duplicate rows across all 12 columns.
- 38 rows share identical coordinates, forming 18 groups. 14 groups contain exactly one row with an `OBJECTID`; 4 groups contain none.
- 7 of the 14 groups have the same operator, charger type and plug count, and differ only in the rating format (e.g. `22 kW` vs `22`) and the address format.
- In the other 7 groups, the rows differ in operator, rating, charger type or plug count, e.g.
  - (-34.4366, 150.8633): `Tesla` / DC / `175 kW` vs `Tesla Motors ` / DC / `2x350kW & 2x175kW`
  - (-34.4080, 150.8771): `Chargefox` / DC / `175 kW` vs `University of` / DC / `2x350kW & 2x175kW`
  - (-33.8556, 151.0768): `Engie` / DC vs `Tesla Motors ` / `Upcoming`
  - (-36.0806, 146.9216): `Exploren` / AC / `AC` / 4 plugs vs `Exploren` / AC / `7` / 2 plugs
- The 4 groups without an `OBJECTID` hold different operators or charger types at the same coordinates (e.g. `Non-networked` AC and `Tesla` DC), except (-32.7166, 151.2573): two `Non-networked` AC rows with 1 and 2 plugs.

## Part 3: SA4 boundary data profile

Profile of `SA4_2026_AUST_GDA2020.shp` (ABS ASGS Edition 4).

In [31]:
sa4 = gpd.read_file("../data/raw/SA4_2026_AUST_SHP_GDA2020/SA4_2026_AUST_GDA2020.shp")
print("Shape:", sa4.shape)
print("CRS:", sa4.crs)
print(sa4.dtypes)
print()
print("Missing attribute values:")
print(sa4.drop(columns="geometry").isna().sum())

Shape: (108, 12)
CRS: EPSG:7844
SA4_CODE26         str
SA4_NAME26         str
CHG_FLAG26         str
CHG_LBL26          str
GCC_CODE26         str
GCC_NAME26         str
STE_CODE26         str
STE_NAME26         str
AUS_CODE26         str
AUS_NAME26         str
AREASQKM26     float64
geometry      geometry
dtype: object

Missing attribute values:
SA4_CODE26     0
SA4_NAME26     0
CHG_FLAG26     0
CHG_LBL26      0
GCC_CODE26     0
GCC_NAME26     0
STE_CODE26     0
STE_NAME26     0
AUS_CODE26     0
AUS_NAME26     0
AREASQKM26    19
dtype: int64


In [32]:
has_geom = sa4.geometry.notna()
print("Null geometries:", (~has_geom).sum())
print("Invalid geometries:", (~sa4[has_geom].is_valid).sum())
print("Geometry types:", sa4[has_geom].geom_type.value_counts().to_dict())
print("Duplicate SA4 codes:", sa4["SA4_CODE26"].duplicated().sum())
print("Duplicate SA4 names:", sa4["SA4_NAME26"].duplicated().sum())
print()
print("Rows without geometry:")
sa4.loc[~has_geom, ["SA4_CODE26", "SA4_NAME26", "STE_NAME26", "AREASQKM26"]]

Null geometries: 19


Invalid geometries: 0
Geometry types: {'MultiPolygon': 48, 'Polygon': 41}
Duplicate SA4 codes: 0
Duplicate SA4 names: 0

Rows without geometry:


,SA4_CODE26,SA4_NAME26,STE_NAME26,AREASQKM26
28,197,Migratory - Offshore - Shipping (NSW),New South Wales,NaN
29,199,No usual address (NSW),New South Wales,NaN
47,297,Migratory - Offshore - Shipping (Vic),Victoria,NaN
48,299,No usual address (Vic),Victoria,NaN
68,397,Migratory - Offshore - Shipping (Qld),Queensland,NaN
69,399,No usual address (Qld),Queensland,NaN
77,497,Migratory - Offshore - Shipping (SA),South Australia,NaN
78,499,No usual address (SA),South Australia,NaN
89,597,Migratory - Offshore - Shipping (WA),Western Australia,NaN
90,599,No usual address (WA),Western Australia,NaN


In [33]:
pd.DataFrame({
    "rows": sa4["STE_NAME26"].value_counts(),
    "with_geometry": sa4[has_geom]["STE_NAME26"].value_counts(),
}).fillna(0).astype(int)

,rows,with_geometry
STE_NAME26,,
Australian Capital Territory,3,1
New South Wales,30,28
Northern Territory,4,2
Other Territories,3,1
Outside Australia,1,0
Queensland,21,19
South Australia,9,7
Tasmania,6,4
Victoria,19,17


In [34]:
sa4["CHG_LBL26"].value_counts()

CHG_LBL26
No change      104
Name change      4
Name: count, dtype: int64

### 3.1 Spatial join coverage

Uses `joined` from Part 1 after the nearest-SA4 fix for the single unmatched point.

In [35]:
print("Charger rows:", len(joined))
print("Rows without SA4:", joined["SA4_CODE26"].isna().sum())
print("Distinct SA4 regions with chargers:", joined["SA4_CODE26"].nunique())
print("NSW SA4 regions with geometry:",
      ((sa4["STE_NAME26"] == "New South Wales") & has_geom).sum())
print("States assigned:", joined["STE_NAME26"].value_counts().to_dict())
joined["SA4_NAME26"].value_counts()

Charger rows: 1958
Rows without SA4: 0
Distinct SA4 regions with chargers: 28
NSW SA4 regions with geometry: 28
States assigned: {'New South Wales': 1958}


SA4_NAME26
Sydney - Eastern Suburbs                  221
Sydney - City and Inner South             139
Hunter Valley exc Newcastle               128
Capital Region                            117
Sydney - Inner West                       116
Central West                              113
Sydney - North Sydney and Hornsby         109
Newcastle and Lake Macquarie               87
Richmond - Tweed                           76
Southern Highlands and Shoalhaven          76
Riverina                                   73
Murray                                     70
Illawarra                                  65
Mid North Coast                            64
New England and North West                 61
Sydney - Northern Beaches                  59
Sydney - Outer West and Blue Mountains     46
Central Coast                              44
Sydney - Inner South West                  43
Sydney - Blacktown                         39
Sydney - Parramatta                        34
Sydney - South West    

## Part 4: Summary of observed data quality issues

| # | Field | Observation |
|---|---|---|
| 1 | `OBJECTID` | Empty in 1837 of 1958 rows; the 121 non-empty rows are contiguous (row index 983-1103) |
| 2 | `LGANAME`, `PCODE`, `Source` | Each empty in exactly 121 rows: the same rows that have an `OBJECTID` |
| 3 | `Charger_Type` | Three values: `AC`, `DC`, `Upcoming` |
| 4 | `Charger_rating` | Mixed formats: `N kW`, number only, the literal `AC`, and combined strings such as `2x350kW & 2x175kW` |
| 5 | `Operator` | 50 distinct values including trailing spaces, case variants and alternative or shortened spellings of the same operator |
| 6 | `Station_name` | Empty in 1438 rows (almost all DC rows); surrounding spaces; some values identical to `Operator` |
| 7 | `Station_address` | Mixed separators (733 rows use line breaks); 1954 of 1958 contain a 4-digit number starting with 2; 2 have no postcode, 2 have postcode 3644; 1 has an empty street part |
| 8 | `PCODE` | Mostly 4 digits; 10 values prefixed with `NSW ` |
| 9 | `Number_of_plugs` | Complete, range 1-35 |
| 10 | Coordinates | Complete and inside NSW; 18 groups of rows share identical coordinates; 0 exact duplicate rows |
| 11 | SA4 boundaries | 108 regions, 19 without geometry (non-spatial categories); no invalid geometries; all chargers assigned to one of the 28 NSW SA4 regions with geometry |


## Part 5: Before and after cleaning

Compares the raw file with the output of `src/03_clean_data.py` (`data/processed/ev_clean.csv`). Run step 03 before this part. The cleaning rules and their reasons are documented in the script.

In [36]:
clean = pd.read_csv("../data/processed/ev_clean.csv", dtype=str, keep_default_na=False)
print("Raw shape:  ", raw.shape)
print("Clean shape:", clean.shape)
print("Columns removed:", sorted(set(raw.columns) - set(clean.columns)))
print("Columns added:  ", sorted(set(clean.columns) - set(raw.columns)))

Raw shape:   (1958, 12)
Clean shape: (1951, 11)
Columns removed: ['OBJECTID']
Columns added:   []


### 5.1 Completeness and formatting

In [37]:
before = text_profile(raw)
after = text_profile(clean)
comparison = before[["empty", "distinct_non_empty", "leading_trailing_space", "contains_newline"]].join(
    after[["empty", "distinct_non_empty", "leading_trailing_space", "contains_newline"]],
    lsuffix="_before", rsuffix="_after", how="left")
comparison

,empty_before,distinct_non_empty_before,leading_trailing_space_before,contains_newline_before,empty_after,distinct_non_empty_after,leading_trailing_space_after,contains_newline_after
column,,,,,,,,
OBJECTID,1837,121,0,0,NaN,NaN,NaN,NaN
Station_name,1438,477,20,0,1431.0,476.0,0.0,0.0
Station_address,0,1924,3,733,0.0,1908.0,0.0,0.0
Operator,0,50,55,0,0.0,42.0,0.0,0.0
Number_of_plugs,0,18,0,0,0.0,18.0,0.0,0.0
Charger_Type,0,3,0,0,0.0,3.0,0.0,0.0
Charger_rating,0,46,0,0,522.0,42.0,0.0,0.0
Latitude,0,1937,0,0,0.0,1937.0,0.0,0.0
Longitude,0,1935,0,0,0.0,1935.0,0.0,0.0


### 5.2 Operator

In [38]:
print("Distinct operators:", raw["Operator"].nunique(), "->", clean["Operator"].nunique())
clean["Operator"].value_counts().sort_index()

Distinct operators: 50 -> 42


Operator
360 ev charge                     5
alchemy charge                    1
ampol                            32
axcharge                          1
bmw                               1
bp                               60
casacharge                        5
charge os                         2
chargefox                       253
chargehub                        18
chargepoint                       8
chargepost                        5
chargestar                        2
counties energy                   1
elanga                            7
energy austra                     1
engie                             6
ev meter                          1
eve australia                    22
everty                           36
evie networks                   103
evnet                            16
evse                             18
evup                             38
evx                             109
exploren                        301
fast cities a                    17
gentari            

### 5.3 Charger_rating

In [39]:
def rating_format_or_null(value):
    return "NULL" if value == "" else rating_format(value)

pd.DataFrame({
    "before": raw["Charger_rating"].map(rating_format_or_null).value_counts(),
    "after": clean["Charger_rating"].map(rating_format_or_null).value_counts(),
}).fillna(0).astype(int)

,before,after
Charger_rating,,
AC,522,0
NULL,0,522
combined (e.g. 2x350kW & 2x175kW),99,99
number + ' kW',1315,1330
number only,22,0


### 5.4 Station_address and PCODE

In [40]:
for name, df_ in [("before", raw), ("after", clean)]:
    addr = df_["Station_address"]
    pcode = df_["PCODE"]
    print(f"{name:>6}: addresses with line breaks = {addr.str.contains(chr(10)).sum()}, "
          f"repeated commas = {addr.str.contains(',,').sum()}, "
          f"PCODE empty = {(pcode == '').sum()}, "
          f"PCODE not 4 digits = {((pcode != '') & ~pcode.str.fullmatch(r'\d{4}')).sum()}")

before: addresses with line breaks = 733, repeated commas = 1, PCODE empty = 121, PCODE not 4 digits = 10
 after: addresses with line breaks = 0, repeated commas = 0, PCODE empty = 0, PCODE not 4 digits = 0


### 5.5 Charger types and rows at identical coordinates

In [41]:
print(pd.DataFrame({
    "before": raw["Charger_Type"].value_counts(),
    "after": clean["Charger_Type"].value_counts(),
}))
print()
for name, df_ in [("before", raw), ("after", clean)]:
    shared_ = df_[df_.duplicated(coord_cols, keep=False)]
    print(f"{name:>6}: {len(shared_)} rows in {shared_.groupby(coord_cols).ngroups} groups share coordinates")

              before  after
Charger_Type               
AC              1427   1421
DC               433    432
Upcoming          98     98

before: 38 rows in 18 groups share coordinates
 after: 24 rows in 11 groups share coordinates


### 5.6 Unresolved issue: address and coordinates of different towns

The rows below have a `PCODE` that differs from the postcode written in their address. For most of them, the coordinates and `PCODE` belong to one town while `Station_address` and `LGANAME` belong to another (e.g. address in Wilcannia 2836, `PCODE` 2350 and coordinates in Armidale). The file gives no reliable way to decide which fields are correct, so these rows are left unchanged. The SA4 region is assigned from the coordinates.

In [42]:
address_pc = clean["Station_address"].str.findall(r"\b(\d{4})\b").str[-1]
differs = (clean["PCODE"] != "") & address_pc.notna() & (clean["PCODE"] != address_pc)
print("Rows where PCODE differs from the address postcode:", differs.sum())
clean.loc[differs, ["Operator", "Charger_Type", "Station_address", "PCODE", "LGANAME",
                    "Latitude", "Longitude"]].assign(address_postcode=address_pc[differs])

Rows where PCODE differs from the address postcode: 25


,Operator,Charger_Type,Station_address,PCODE,LGANAME,Latitude,Longitude,address_postcode
2,nrma electric,DC,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",2350,Central Darling Shire Council,-30.5118739,151.669395,2836
27,nrma electric,DC,"1 Little Walker St, Casino, NSW 2470, Australia",2622,Richmond Valley Council,-35.442745,149.7913305,2470
76,nrma electric,DC,"10W Apsley St, Walcha NSW 2354, Australia",2839,Walcha Council,-29.960127,146.8578887,2354
94,nrma electric,DC,"116 Liverpool St, Scone, NSW 2337, Australia",2880,Upper Hunter Shire Council,-31.9598511,141.4601038,2337
179,nrma electric,DC,"15 Victory St, Braidwood, NSW 2622, Australia",2835,Queanbeyan-Palerang Regional Council,-31.4988197,145.8375768,2622
193,nrma electric,DC,"157 Rouse Street, Tenterfield, NSW 2372, Austr...",2825,Tenterfield Shire Council,-31.5645996,147.1969355,2372
195,nrma electric,DC,"15c Mitchell St, Bourke, NSW 2840, Australia",2453,Bourke Shire Council,-30.3383734,152.7125414,2840
215,nrma electric,DC,"17 Stewart St, Wollongong NSW 2500, Australia",2453,Wollongong City Council,-29.7756503,151.1151033,2500
224,nrma electric,DC,"18 Dandaloo St, Nyngan, NSW 2825, Australia",2360,Bogan Shire Council,-32.0506213,150.8667598,2825
331,evie networks,DC,"221 Wolseley St, Jamisontown NSW 2750",2614,Penrith City Council,-33.770295,150.673823,2750
